# Data Preparation Pipeline for Spikeball / Roundnet Datasets

This notebook describes and implements the data preparation pipeline for merging and cleaning multiple public Spikeball/Roundnet datasets. The goal is to create a unified dataset for object detection tasks (Spikeball, Net, and Person with Ball).

In [1]:
# Import required libraries
import os
import shutil
import sys
import pandas as pd
sys.path.append('./test_v2')  # Pour importer utils.py si besoin
from utils import gather_images_and_labels

In [2]:
# Nettoyage initial : suppression des fichiers .txt dans tous les sous-dossiers
from utils import delete_txt_files_in_subfolders

# Supprimer les fichiers .txt dans les dossiers de datasets
root_path = "./DATASET/"  # ou spécifiez le chemin vers vos datasets
delete_txt_files_in_subfolders(root_path)


📁 Scan du dossier : dataset_final_split
📁 Scan du dossier : dataset_by_video
📁 Scan du dossier : dataset_merged
📁 Scan du dossier : dataset_merged_clean
📁 Scan du dossier : ORIGIN

✅ Suppression terminée. 0 fichiers .txt supprimés au total.


## Step 1: Gather all images and labels into a single folder

We will use the function `gather_images_and_labels` to collect all images and labels from the 'train' subfolders of each dataset into a new destination folder. This ensures all data is centralized for further processing.

In [3]:
# Define source and destination paths
root_path = '.'  # Adapt this path to where your datasets folders are
output_folder = './DATASET/dataset_merged'  # Destination folder for all images and labels

# Gather all images and labels
# This will create the output_folder and copy all files into it
gather_images_and_labels(root_path, output_folder)

## Step 2: Merge datasets and remove duplicates

After standardizing the class orders in all datasets, we can now merge them safely while removing duplicates. Many images appear in multiple datasets with different names (often with Roboflow's ".rf." format). We'll use the `merge_datasets` function to:
- Detect duplicates based on base filename (before ".rf.")
- Give priority to images from "Roundnet Project" dataset (which has the "Person with Ball" label)
- Keep only one version of each unique image

## Step 2a: Remap YOLO labels to ensure consistent class order

Before merging datasets, we need to ensure all datasets use the same class order. Different datasets might have different class orders in their data.yaml files, which could cause issues during training. We'll use the `remap_yolo_labels_inplace` function to standardize the class order across all datasets.

In [4]:
# Import the remap function
from utils import remap_yolo_labels_inplace

# Define the standardized class order for Spikeball/Roundnet
# This ensures consistency across all datasets
standard_class_order = ['Net', 'Spikeball']

print("🔄 REMAPPING YOLO LABELS TO STANDARDIZED CLASS ORDER")
print("=" * 60)
print(f"Standard class order: {standard_class_order}")
print()

# Define the paths to the original datasets
dataset_dirs = [
    "DATASET/ORIGIN/Roundnet AI",
    "DATASET/ORIGIN/Roundnet Project", 
    "DATASET/ORIGIN/Spikeball RoundnetAI"
]

# Remap each dataset - remap_yolo_labels_inplace handles everything including validation
for i, dataset_dir in enumerate(dataset_dirs, 1):
    print(f"📂 {i}. Processing dataset: {dataset_dir}")
    
    if not os.path.exists(dataset_dir):
        print(f"   ❌ Dataset not found, skipping...")
        continue
    
    # Check if data.yaml exists
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    if not os.path.exists(yaml_path):
        print(f"   ⚠️ No data.yaml found, skipping...")
        continue
    
    try:
        # Apply the remapping - this function now handles all validation internally
        result = remap_yolo_labels_inplace(dataset_dir, standard_class_order)
        print(f"   {result}")
        
    except Exception as e:
        print(f"   ❌ Error during remapping: {e}")
    
    print()

print("✅ YOLO label remapping completed for all datasets!")
print("All datasets now use the same standardized class order.")
print(f"Ready for merge step with consistent labels.")

🔄 REMAPPING YOLO LABELS TO STANDARDIZED CLASS ORDER
Standard class order: ['Net', 'Spikeball']

📂 1. Processing dataset: DATASET/ORIGIN/Roundnet AI
✅ L'ordre des classes dans data.yaml est déjà correct.
🔍 Vérification des fichiers pour d'éventuelles incohérences...
✅ Aucune incohérence détectée, tout est conforme.
   ✅ Aucune modification nécessaire : DATASET/ORIGIN/Roundnet AI

📂 2. Processing dataset: DATASET/ORIGIN/Roundnet Project
✅ L'ordre des classes dans data.yaml est déjà correct.
🔍 Vérification des fichiers pour d'éventuelles incohérences...
⚠️ Incohérence détectée ! 551 lignes avec des IDs invalides supprimées de 518 fichiers.
   ✅ Incohérences corrigées dans : DATASET/ORIGIN/Roundnet Project

📂 3. Processing dataset: DATASET/ORIGIN/Spikeball RoundnetAI
✅ L'ordre des classes dans data.yaml est déjà correct.
🔍 Vérification des fichiers pour d'éventuelles incohérences...
✅ Aucune incohérence détectée, tout est conforme.
   ✅ Aucune modification nécessaire : DATASET/ORIGIN/Spike

In [5]:
# Import the merge function
from utils import merge_datasets

# Define the paths to the original datasets
dataset_dirs = [
    "DATASET/ORIGIN/Roundnet AI",
    "DATASET/ORIGIN/Roundnet Project", 
    "DATASET/ORIGIN/Spikeball RoundnetAI"
]

# Define output directory for merged dataset
merged_output_dir = "./DATASET/dataset_merged_clean"

print("Datasets to merge:")
for i, dataset in enumerate(dataset_dirs, 1):
    print(f"{i}. {dataset}")
    if os.path.exists(dataset):
        train_images = os.path.join(dataset, "train/images")
        if os.path.exists(train_images):
            count = len([f for f in os.listdir(train_images) if f.endswith(('.jpg', '.png'))])
            print(f"   → {count} images found")
        else:
            print(f"   → No train/images folder found")
    else:
        print(f"   → Dataset not found!")

Datasets to merge:
1. DATASET/ORIGIN/Roundnet AI
   → 1237 images found
2. DATASET/ORIGIN/Roundnet Project
   → 861 images found
3. DATASET/ORIGIN/Spikeball RoundnetAI
   → 806 images found


In [6]:
# Vérifier la valeur de merged_output_dir
print(f"merged_output_dir = {merged_output_dir}")
print(f"Le dossier existe-t-il ? {os.path.exists(merged_output_dir)}")

if os.path.exists(merged_output_dir):
    images_dir = os.path.join(merged_output_dir, "images")
    labels_dir = os.path.join(merged_output_dir, "labels")
    
    if os.path.exists(images_dir):
        image_count = len([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        print(f"Images actuelles dans {images_dir}: {image_count}")
    
    if os.path.exists(labels_dir):
        label_count = len([f for f in os.listdir(labels_dir) if f.endswith('.txt')])
        print(f"Labels actuels dans {labels_dir}: {label_count}")

merged_output_dir = ./DATASET/dataset_merged_clean
Le dossier existe-t-il ? True
Images actuelles dans ./DATASET/dataset_merged_clean/images: 1074
Labels actuels dans ./DATASET/dataset_merged_clean/labels: 1074


In [7]:
# Execute the merge process
print("🚀 Starting dataset merge process...")
print(f"Priority dataset: 'Roundnet Project' (has Person with Ball labels)")
print(f"Output directory: {merged_output_dir}")
print("-" * 50)

# Run the merge function
merge_datasets(
    dataset_dirs=dataset_dirs,
    output_dir=merged_output_dir,
    priority_dataset="Roundnet Project"
)

print("-" * 50)
print("✅ Merge process completed!")

🚀 Starting dataset merge process...
Priority dataset: 'Roundnet Project' (has Person with Ball labels)
Output directory: ./DATASET/dataset_merged_clean
--------------------------------------------------
🧩 1076 groupes d’images uniques détectés (via nom avant '.rf')

✅ Fusion terminée. 1076 images retenues et copiées dans ./DATASET/dataset_merged_clean
--------------------------------------------------
✅ Merge process completed!

✅ Fusion terminée. 1076 images retenues et copiées dans ./DATASET/dataset_merged_clean
--------------------------------------------------
✅ Merge process completed!


In [8]:
# Verify the merged dataset
if os.path.exists(merged_output_dir):
    images_dir = os.path.join(merged_output_dir, "images")
    labels_dir = os.path.join(merged_output_dir, "labels")
    
    if os.path.exists(images_dir) and os.path.exists(labels_dir):
        # Count files
        image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
        label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
        
        print("📊 MERGED DATASET STATISTICS")
        print("=" * 40)
        print(f"Total images: {len(image_files)}")
        print(f"Total labels: {len(label_files)}")
        print(f"Images with labels: {len([img for img in image_files if os.path.splitext(img)[0] + '.txt' in label_files])}")
        print(f"Images without labels: {len(image_files) - len([img for img in image_files if os.path.splitext(img)[0] + '.txt' in label_files])}")
        
        # Show some examples
        print(f"\nFirst 5 image files:")
        for i, img in enumerate(image_files[:5], 1):
            print(f"  {i}. {img}")
            
    else:
        print("❌ Images or labels directory not found in merged dataset")
else:
    print("❌ Merged dataset directory not found")

📊 MERGED DATASET STATISTICS
Total images: 1076
Total labels: 1076
Images with labels: 1076
Images without labels: 0

First 5 image files:
  1. IMG_1501-1-_f_2870_jpg.rf.11f900929a00edda4f7c189de45eaf37.jpg
  2. IMG_1578_f_2057_jpg.rf.456f01d79f6a6c414c69aaf9b83d5a0b.jpg
  3. independent-serving_f_4295_jpg.rf.966399f4666ac55244165cb54eb7eee5.jpg
  4. VID_20201204_155614_f_10457_jpg.rf.0ad20151b1c9080839c670060963ee9a.jpg
  5. IMG_1501-1-_f_1061_jpg.rf.b1ed93ff0f5e02a48c0613ccdc80fd15.jpg


In [9]:
# Clean up images without labels and empty labels
from utils import remove_images_without_labels

print("🧹 CLEANING IMAGES WITHOUT VALID LABELS")
print("=" * 50)

# Clean the merged dataset by removing images without corresponding labels or with empty labels
cleaning_stats = remove_images_without_labels(merged_output_dir)

if cleaning_stats and cleaning_stats['removal_successful']:
    print(f"\n📊 CLEANING SUMMARY:")
    print(f"   🔄 Total images processed: {cleaning_stats['total_images']}")
    print(f"   📄 Total labels found: {cleaning_stats['total_labels']}")
    print(f"   ✅ Valid labels (non-empty): {cleaning_stats['valid_labels']}")
    print(f"   ❌ Empty labels found: {cleaning_stats['empty_labels']}")
    print(f"   ✅ Images with valid labels: {cleaning_stats['images_with_valid_labels']}")
    print(f"   🗑️ Images removed: {cleaning_stats['images_removed']}")
    print(f"   🗑️ Empty labels removed: {cleaning_stats['empty_labels_removed']}")
    
    if cleaning_stats['images_removed'] > 0 or cleaning_stats['empty_labels_removed'] > 0:
        print(f"\n💡 Dataset is now cleaner:")
        if cleaning_stats['images_removed'] > 0:
            print(f"   - Removed {cleaning_stats['images_removed']} images without valid labels")
        if cleaning_stats['empty_labels_removed'] > 0:
            print(f"   - Removed {cleaning_stats['empty_labels_removed']} empty label files")
    else:
        print(f"\n✨ Dataset was already clean - no orphaned images or empty labels found!")
else:
    print("❌ Error occurred during cleaning process")

🧹 CLEANING IMAGES WITHOUT VALID LABELS
🔍 ANALYSE DU DATASET: ./DATASET/dataset_merged_clean
   📊 Total images: 1076
   📄 Total labels: 1076
   ✅ Labels valides (non vides): 1074
   ❌ Labels vides: 2
   ✅ Images avec labels valides: 1074
   ❌ Images sans labels valides: 2

🗑️ Suppression de 2 fichiers de labels vides...
   🗑️ Label vide supprimé: IMG_1501-1-_f_2051_jpg.rf.e2177ca3b54bc74a28e8bd4a2c53e47d.txt
   🗑️ Label vide supprimé: IMG_1826_f_14653_jpg.rf.0475f5a6413d8163a89aceb105905d25.txt

🗑️ Suppression de 2 images sans labels valides...
   🗑️ Image supprimée: IMG_1501-1-_f_2051_jpg.rf.e2177ca3b54bc74a28e8bd4a2c53e47d.jpg
   🗑️ Image supprimée: IMG_1826_f_14653_jpg.rf.0475f5a6413d8163a89aceb105905d25.jpg

✅ Nettoyage terminé:
   🗑️ 2/2 images supprimées
   🗑️ 2/2 labels vides supprimés

📊 CLEANING SUMMARY:
   🔄 Total images processed: 1076
   📄 Total labels found: 1076
   ✅ Valid labels (non-empty): 1074
   ❌ Empty labels found: 2
   ✅ Images with valid labels: 1074
   🗑️ Images 

## Step 3: Organize data by video source

Now that we have a clean merged dataset, we can organize the images and labels by their video source using the filename prefix. This will create separate folders for each video, making it easier to analyze data distribution and potentially split the dataset for training/validation based on video sources.

In [10]:
# Import the sort function
from utils import sort_images_by_video
import re

# Define the source directory (merged dataset) and destination
source_dir = merged_output_dir  # Le dossier principal qui contient images/ et labels/
video_organized_dir = "./DATASET/dataset_by_video"

print("📁 Organizing images and labels by video source...")
print(f"Source: {source_dir}")
print(f"Destination: {video_organized_dir}")
print("-" * 50)

# Vérifier le contenu du dataset nettoyé
images_source = os.path.join(source_dir, "images")
labels_source = os.path.join(source_dir, "labels")

if not os.path.exists(images_source) or not os.path.exists(labels_source):
    print("❌ Source directories not found!")
    print(f"   Images: {images_source} (exists: {os.path.exists(images_source)})")
    print(f"   Labels: {labels_source} (exists: {os.path.exists(labels_source)})")
else:
    # Compter les fichiers après nettoyage
    image_files = [f for f in os.listdir(images_source) if f.endswith(('.jpg', '.png', '.jpeg'))]
    label_files = [f for f in os.listdir(labels_source) if f.endswith('.txt')]
    
    print(f"📊 Dataset after cleaning:")
    print(f"   Images: {len(image_files)}")
    print(f"   Labels: {len(label_files)}")
    print(f"   ✅ All images should have valid labels after cleaning step")

    # Supprimer le dossier de destination s'il existe déjà pour éviter les doublons
    if os.path.exists(video_organized_dir):
        shutil.rmtree(video_organized_dir)
    os.makedirs(video_organized_dir, exist_ok=True)
    print("🧹 Destination folder cleaned")

    # Organisation par vidéo - Version simplifiée utilisant directement sort_images_by_video
    # mais en passant par un dossier temporaire pour éviter les problèmes de structure
    temp_dir = "./temp_for_video_sort"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)

    # Copier TOUS les fichiers du dataset nettoyé (ils sont tous valides)
    copied_images = 0
    copied_labels = 0

    print(f"\n📂 Copying cleaned dataset files to temporary folder...")
    
    # Copier toutes les images
    for img_file in image_files:
        shutil.copy2(os.path.join(images_source, img_file), os.path.join(temp_dir, img_file))
        copied_images += 1
    
    # Copier tous les labels
    for lbl_file in label_files:
        shutil.copy2(os.path.join(labels_source, lbl_file), os.path.join(temp_dir, lbl_file))
        copied_labels += 1

    print(f"   📸 Images copied: {copied_images}")
    print(f"   🏷️ Labels copied: {copied_labels}")

    # Utiliser la fonction sort_images_by_video existante
    print(f"\n📁 Organizing by video using sort_images_by_video...")
    sort_images_by_video(temp_dir, video_organized_dir)

    # Nettoyer le dossier temporaire
    shutil.rmtree(temp_dir)
    print("🧹 Cleaned up temporary directory")

    print(f"\n✅ Video organization completed!")
    print(f"   📂 All files from cleaned dataset organized by video source")

📁 Organizing images and labels by video source...
Source: ./DATASET/dataset_merged_clean
Destination: ./DATASET/dataset_by_video
--------------------------------------------------
📊 Dataset after cleaning:
   Images: 1074
   Labels: 1074
   ✅ All images should have valid labels after cleaning step
🧹 Destination folder cleaned

📂 Copying cleaned dataset files to temporary folder...
🧹 Destination folder cleaned

📂 Copying cleaned dataset files to temporary folder...
   📸 Images copied: 1074
   🏷️ Labels copied: 1074

📁 Organizing by video using sort_images_by_video...
📁 Tri des fichiers depuis './temp_for_video_sort' vers './DATASET/dataset_by_video'...
✅ Label copié : Game1KevinRyanJosh_f_841_jpg.rf.8e885951423799c14bd8a06c6c7c9590.txt → Game1KevinRyanJosh_f/labels/
✅ Label copié : IMG_1547_f_2038_jpg.rf.94fadd4c23ce2a2a5613c920bf56c510.txt → IMG_1547_f/labels/
✅ Label copié : IMG_4918_f_10286_jpg.rf.ef9ef696d1c3550bc919d6f0cf053fc6.txt → IMG_4918_f/labels/
✅ Label copié : IMG_1696_f_13

In [11]:
# Verify the video-organized dataset
if os.path.exists(video_organized_dir):
    video_folders = [f for f in os.listdir(video_organized_dir) if os.path.isdir(os.path.join(video_organized_dir, f))]
    
    print("📊 VIDEO-ORGANIZED DATASET STATISTICS")
    print("=" * 50)
    print(f"Total video folders created: {len(video_folders)}")
    print("\nVideo folders and their content:")
    
    total_images = 0
    total_labels = 0
    
    for i, video_folder in enumerate(sorted(video_folders), 1):
        video_path = os.path.join(video_organized_dir, video_folder)
        images_path = os.path.join(video_path, "images")
        labels_path = os.path.join(video_path, "labels")
        
        img_count = len(os.listdir(images_path)) if os.path.exists(images_path) else 0
        lbl_count = len(os.listdir(labels_path)) if os.path.exists(labels_path) else 0
        
        total_images += img_count
        total_labels += lbl_count
        
        print(f"  {i:2d}. {video_folder}")
        print(f"      └── Images: {img_count}, Labels: {lbl_count}")
    
    print(f"\n📈 TOTAL: {total_images} images, {total_labels} labels across {len(video_folders)} videos")
else:
    print("❌ Video-organized dataset directory not found")

📊 VIDEO-ORGANIZED DATASET STATISTICS
Total video folders created: 43

Video folders and their content:
   1. 1_1_f
      └── Images: 12, Labels: 12
   2. Game-1-Part-1_f
      └── Images: 15, Labels: 15
   3. Game1KevinJoshRyanNeil_f
      └── Images: 18, Labels: 18
   4. Game1KevinRyanJosh_f
      └── Images: 20, Labels: 20
   5. Game1RyanRyanJoshBarrett_f
      └── Images: 32, Labels: 32
   6. IMG_1482-1-_f
      └── Images: 22, Labels: 22
   7. IMG_1501-1-_f
      └── Images: 26, Labels: 26
   8. IMG_1524-1-_f
      └── Images: 15, Labels: 15
   9. IMG_1537_f
      └── Images: 12, Labels: 12
  10. IMG_1547_f
      └── Images: 38, Labels: 38
  11. IMG_1560_f
      └── Images: 32, Labels: 32
  12. IMG_1566_f
      └── Images: 12, Labels: 12
  13. IMG_1578_f
      └── Images: 25, Labels: 25
  14. IMG_1588_f
      └── Images: 22, Labels: 22
  15. IMG_1597_f
      └── Images: 25, Labels: 25
  16. IMG_1633_f
      └── Images: 24, Labels: 24
  17. IMG_1641_f
      └── Images: 31, Labels: 3

In [12]:
# Comparaison des totaux avant/après organisation par vidéo
print("🔍 COMPARAISON AVANT/APRÈS ORGANISATION PAR VIDÉO")
print("=" * 55)

# Avant (dataset nettoyé)
images_source = os.path.join(merged_output_dir, "images")
labels_source = os.path.join(merged_output_dir, "labels")

if os.path.exists(images_source):
    before_images = len([f for f in os.listdir(images_source) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"📊 AVANT (dataset nettoyé): {before_images} images")
else:
    before_images = 0
    print("❌ Dossier source d'images introuvable")

if os.path.exists(labels_source):
    before_labels = len([f for f in os.listdir(labels_source) if f.endswith('.txt')])
    print(f"📊 AVANT (dataset nettoyé): {before_labels} labels")
else:
    before_labels = 0
    print("❌ Dossier source de labels introuvable")

# Après (dataset organisé par vidéo)
video_organized_dir = "./DATASET/dataset_by_video"
if os.path.exists(video_organized_dir):
    total_images_after = 0
    total_labels_after = 0
    video_folders = [f for f in os.listdir(video_organized_dir) if os.path.isdir(os.path.join(video_organized_dir, f))]
    
    for video_folder in video_folders:
        video_path = os.path.join(video_organized_dir, video_folder)
        images_path = os.path.join(video_path, "images")
        labels_path = os.path.join(video_path, "labels")
        
        if os.path.exists(images_path):
            total_images_after += len(os.listdir(images_path))
        if os.path.exists(labels_path):
            total_labels_after += len(os.listdir(labels_path))
    
    print(f"📊 APRÈS (organisé par vidéo): {total_images_after} images")
    print(f"📊 APRÈS (organisé par vidéo): {total_labels_after} labels")
    
    # Vérification
    if before_images == total_images_after and before_labels == total_labels_after:
        print("\n✅ PARFAIT: Aucune perte de données lors de l'organisation!")
    else:
        print(f"\n⚠️  ATTENTION: Différence détectée!")
        print(f"   Images: {before_images} → {total_images_after} (différence: {total_images_after - before_images})")
        print(f"   Labels: {before_labels} → {total_labels_after} (différence: {total_labels_after - before_labels})")
else:
    print("❌ Dossier organisé par vidéo introuvable")

🔍 COMPARAISON AVANT/APRÈS ORGANISATION PAR VIDÉO
📊 AVANT (dataset nettoyé): 1074 images
📊 AVANT (dataset nettoyé): 1074 labels
📊 APRÈS (organisé par vidéo): 1074 images
📊 APRÈS (organisé par vidéo): 1074 labels

✅ PARFAIT: Aucune perte de données lors de l'organisation!


## Step 5: Dataset Splitting Optimization

Now we'll execute the optimized dataset splitting using our cleaned `run_random_split_optimization` function. This will split the videos into train/validation/test sets while maintaining optimal distribution of objects (balls and nets) across all splits.

### Splitting Strategy:
- **80% for training** - approximately 27-37 videos
- **10% for validation** - approximately 2-9 videos  
- **10% for testing** - remaining videos

The optimization will find the best assignment to minimize the error between target and actual object distributions.

In [13]:
# Execute the optimized dataset splitting WITH IMMEDIATE PERSISTENCE
print("🚀 STARTING DATASET SPLITTING OPTIMIZATION WITH IMMEDIATE PERSISTENCE")
print("=" * 75)

import sys
import os
import json
import importlib
sys.path.append('.')  # Pour importer split_data.py

try:
    # Recharger le module pour prendre en compte les modifications
    if 'split_data' in sys.modules:
        importlib.reload(sys.modules['split_data'])
    
    from split_data import run_random_split_optimization, prepare_dataset, compute_video_count_range_for_split
    print("✅ Successfully imported split_data functions (reloaded)")
except ImportError as e:
    print(f"❌ Error importing split_data: {e}")
    print("   Make sure split_data.py is in the current directory")
    raise

# Prepare the dataset for splitting
print("\n📂 Preparing dataset...")
df, target = prepare_dataset('./DATASET/dataset_by_video')

# Display initial information
total_videos = len(df)
object_totals = {
    'Spikeball': df['Spikeball'].sum(),
    'Net': df['Net'].sum()
}

print(f"\n📊 Dataset loaded:")
print(f"- {total_videos} videos")
print(f"- {object_totals['Spikeball']} balls")
print(f"- {object_totals['Net']} nets")

# Target distribution - Now using standard 'val' key
print(f"\n🎯 Target distribution:")
print(f"- Train (80%): {target['train'][0]:.0f} balls, {target['train'][1]:.0f} nets")
print(f"- Val (10%): {target['val'][0]:.0f} balls, {target['val'][1]:.0f} nets") 
print(f"- Test (10%): {target['test'][0]:.0f} balls, {target['test'][1]:.0f} nets")

# Calculate optimal ranges automatically for train and val
print(f"\n🔢 Calculating optimal ranges...")

# Range pour train (80%)
train_min, train_max = compute_video_count_range_for_split(
    df, 0.8, object_totals, ['Spikeball', 'Net']
)
train_range = (train_min, train_max)

# Range pour val (10%) 
val_min, val_max = compute_video_count_range_for_split(
    df, 0.1, object_totals, ['Spikeball', 'Net']
)
val_range = (val_min, val_max)

print(f"\n⚙️ Optimization configuration (calculated automatically):")
print(f"- Train range: {train_range} videos")
print(f"- Val range: {val_range} videos")
print(f"- Test range: automatically computed")

🚀 STARTING DATASET SPLITTING OPTIMIZATION WITH IMMEDIATE PERSISTENCE
✅ Successfully imported split_data functions (reloaded)

📂 Preparing dataset...

📊 Dataset loaded:
- 43 videos
- 4327 balls
- 1256 nets

🎯 Target distribution:
- Train (80%): 3462 balls, 1005 nets
- Val (10%): 433 balls, 126 nets
- Test (10%): 433 balls, 126 nets

🔢 Calculating optimal ranges...
  Calcul MIN pour Spikeball (cible: 3462):
    Vidéo 1: +267 → cumul = 267/3462 (7.7%)
    Vidéo 2: +248 → cumul = 515/3462 (14.9%)
    Vidéo 3: +236 → cumul = 751/3462 (21.7%)
    Vidéo 22: +80 → cumul = 3185/3462 (92.0%)
    Vidéo 23: +76 → cumul = 3261/3462 (94.2%)
    Vidéo 24: +75 → cumul = 3336/3462 (96.4%)
    Vidéo 25: +75 → cumul = 3411/3462 (98.5%)
    Vidéo 26: +73 → cumul = 3484/3462 (100.6%)
    ✅ Spikeball dépasse à la vidéo 26, donc min = 25 vidéos
  Calcul MIN pour Net (cible: 1005):
    Vidéo 1: +56 → cumul = 56/1005 (5.6%)
    Vidéo 2: +53 → cumul = 109/1005 (10.8%)
    Vidéo 3: +51 → cumul = 160/1005 (15.9%)

In [14]:
# Lancer l'optimisation avec sauvegarde automatique
print("\n🔄 RUNNING OPTIMIZATION WITH PERSISTENCE...")
print("=" * 50)

# Paramètres
split_file = "./DATASET/best_split.json"
n_iterations = 6000

print(f"\n🔄 Running optimization with automatic save...")
print(f"   💾 Save file: {split_file}")
print(f"   ⏱️  Iterations: {n_iterations:,}")

# Lancer l'optimisation avec sauvegarde automatique intégrée
assignment, best_error = run_random_split_optimization(
    df,
    target,
    n_iterations=n_iterations,
    train_range=train_range,
    val_range=val_range,
    split_file=split_file
)

print(f"\n🏆 OPTIMIZATION COMPLETED!")
print(f"   🔄 Iterations completed: {n_iterations:,}")
print(f"   🥇 Final best error: {best_error:.2f}")
print(f"   💾 Best split available in: {split_file}")
print(f"   ✅ Ready for next steps!")


🔄 RUNNING OPTIMIZATION WITH PERSISTENCE...

🔄 Running optimization with automatic save...
   💾 Save file: ./DATASET/best_split.json
   ⏱️  Iterations: 6,000
   📁 Found previous best split with error: 11.50
🔄 Optimisation par méthode aléatoire...
Cibles: Train=4466, Val=558, Test=558 labels
Ranges utilisés: Train=(27, 38), Val=(2, 9)
Espace de recherche: 576,000 combinaisons totales
(6,000 iterations × 96 combinaisons/iteration)
  Iteration 2000/6000, meilleure erreur: 111
    Efficacité: 84.4% (162,000 évaluations valides)
  Iteration 2000/6000, meilleure erreur: 111
    Efficacité: 84.4% (162,000 évaluations valides)
  Iteration 4000/6000, meilleure erreur: 111
    Efficacité: 84.4% (324,000 évaluations valides)
  Iteration 4000/6000, meilleure erreur: 111
    Efficacité: 84.4% (324,000 évaluations valides)
  Iteration 6000/6000, meilleure erreur: 111
    Efficacité: 84.4% (486,000 évaluations valides)
✅ Optimisation terminée. Meilleure erreur: 111
📈 Statistiques: 486,000 évaluations

In [15]:
# Charger et afficher le meilleur split sauvegardé
print("📁 LOADING BEST SAVED SPLIT")
print("=" * 40)

split_file = "./DATASET/best_split.json"

if os.path.exists(split_file):
    # Charger le split
    with open(split_file, 'r') as f:
        split_data = json.load(f)
    
    print(f"✅ Best split loaded successfully!")
    print(f"   📊 Error: {split_data['error']:.2f}")
    print(f"   📅 Created: {split_data['timestamp']}")
    print(f"   📁 File: {split_file}")
    
    # Utiliser les données du split sauvegardé
    assignment = split_data['assignment']
    best_error = split_data['error']
    
    print(f"\n🎯 BEST SPLIT DETAILS:")
    print(f"   🚂 TRAIN ({split_data['statistics']['train']['video_count']} videos):")
    print(f"      └── {split_data['statistics']['train']['ball_count']} balls ({split_data['statistics']['train']['ball_percentage']:.1f}%)")
    print(f"      └── {split_data['statistics']['train']['net_count']} nets ({split_data['statistics']['train']['net_percentage']:.1f}%)")
    
    print(f"   🔍 VALID ({split_data['statistics']['val']['video_count']} videos):")
    print(f"      └── {split_data['statistics']['val']['ball_count']} balls ({split_data['statistics']['val']['ball_percentage']:.1f}%)")
    print(f"      └── {split_data['statistics']['val']['net_count']} nets ({split_data['statistics']['val']['net_percentage']:.1f}%)")
    
    print(f"   🧪 TEST ({split_data['statistics']['test']['video_count']} videos):")
    print(f"      └── {split_data['statistics']['test']['ball_count']} balls ({split_data['statistics']['test']['ball_percentage']:.1f}%)")
    print(f"      └── {split_data['statistics']['test']['net_count']} nets ({split_data['statistics']['test']['net_percentage']:.1f}%)")
    
    print(f"\n💡 TIP: Vous pouvez relancer l'optimisation ci-dessus.")
    print(f"   Le split ne sera remplacé que si un meilleur score est trouvé!")

else:
    print(f"❌ No saved split found at: {split_file}")
    print(f"   Run the optimization above to create your first best split!")

📁 LOADING BEST SAVED SPLIT
✅ Best split loaded successfully!
   📊 Error: 11.50
   📅 Created: 2025-08-03T15:37:41.664310
   📁 File: ./DATASET/best_split.json

🎯 BEST SPLIT DETAILS:
   🚂 TRAIN (34 videos):
      └── 3463 balls (80.0%)
      └── 1005 nets (80.0%)
   🔍 VALID (6 videos):
      └── 430 balls (9.9%)
      └── 126 nets (10.0%)
   🧪 TEST (3 videos):
      └── 434 balls (10.0%)
      └── 125 nets (10.0%)

💡 TIP: Vous pouvez relancer l'optimisation ci-dessus.
   Le split ne sera remplacé que si un meilleur score est trouvé!


### 🏆 Système de Persistence du Meilleur Split

Le système de persistence fonctionne maintenant parfaitement ! Voici comment il opère :

#### 🔄 **Fonctionnement automatique** :
1. **Première optimisation** : Aucun split sauvegardé → Le résultat devient le nouveau record
2. **Optimisations suivantes** : Compare avec le meilleur score existant
   - ✅ **Meilleur score trouvé** → Sauvegarde le nouveau split
   - 📊 **Pas d'amélioration** → Conserve l'ancien split

#### 📁 **Fichier de sauvegarde** :
- **Emplacement** : `./DATASET/best_split.json`
- **Contenu** : Assignment, score d'erreur, statistiques détaillées, timestamp
- **Persistance** : Survit aux redémarrages du notebook

#### 🚀 **Avantages** :
- ✅ **Aucune perte** : Le meilleur split est toujours préservé
- ✅ **Amélioration continue** : Vous pouvez relancer l'optimisation autant de fois que nécessaire
- ✅ **Traçabilité** : Horodatage et détails de chaque meilleur split
- ✅ **Facilité d'usage** : Utilisez simplement la cellule d'optimisation

#### 💡 **Usage recommandé** :
1. Lancez l'optimisation plusieurs fois avec différents paramètres (`n_iterations`, ranges)
2. Le système garde automatiquement le meilleur résultat
3. Utilisez le split sauvegardé pour créer vos dossiers train/val/test

In [16]:
# Display and analyze the splitting results
from split_data import summarize_distribution


stats = summarize_distribution(df, assignment, target)

print("🎯 FINAL SPLITTING RESULTS")
print("=" * 60)

for split in ['train', 'val', 'test']:
    s = stats[split]
    print(f"\n{split.upper()} SET ({s['video_count']} videos):")
    print(f"  Balls: {s['ball_count']} ({s['ball_percentage']:.1f}% of total)")
    print(f"  Nets: {s['net_count']} ({s['net_percentage']:.1f}% of total)")
    print(f"  Videos: {', '.join(s['videos'][:3])}{'...' if len(s['videos']) > 3 else ''}")


🎯 FINAL SPLITTING RESULTS

TRAIN SET (34 videos):
  Balls: 3463 (80.0% of total)
  Nets: 1005 (80.0% of total)
  Videos: IMG_1578_f, IMG_1524-1-_f, Game1KevinJoshRyanNeil_f...

VAL SET (6 videos):
  Balls: 430 (9.9% of total)
  Nets: 126 (10.0% of total)
  Videos: IMG_1865_f, independent-serving_Trim_f, IMG_1566_f...

TEST SET (3 videos):
  Balls: 434 (10.0% of total)
  Nets: 125 (10.0% of total)
  Videos: independent-serving_f, IMG_1547_f, Game1KevinRyanJosh_f


In [17]:
# Import and execute the folder creation function using BEST SAVED SPLIT
from split_data import create_split_folders

print("📁 CREATING TRAIN/VAL/TEST FOLDER STRUCTURE FROM BEST SPLIT")
print("=" * 65)

# Charger le meilleur split sauvegardé
split_file = "./DATASET/best_split.json"
if os.path.exists(split_file):
    with open(split_file, 'r') as f:
        split_data = json.load(f)
    
    assignment = split_data['assignment']
    print(f"✅ Using best saved split (error: {split_data['error']:.2f})")
    print(f"   📅 Created: {split_data['timestamp']}")
else:
    print(f"❌ No saved split found! Please run the optimization first.")
    print(f"   Expected file: {split_file}")
    exit()

# Execute the folder creation
source_path = './DATASET/dataset_by_video'
output_path = './DATASET/dataset_final_split'

print(f"\nSource: {source_path}")
print(f"Output: {output_path}")
print("Creating folders and copying files...")

copied_counts = create_split_folders(assignment, source_path, output_path)

print(f"\n✅ Split folders created successfully using BEST SPLIT!")
print(f"Files copied:")
for split, count in copied_counts.items():
    print(f"  {split.upper()}: {count} images")

# Verify the final structure
print(f"\n🔍 FINAL DATASET STRUCTURE:")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(output_path, split)
    if os.path.exists(split_path):
        images_count = len(os.listdir(os.path.join(split_path, 'images')))
        labels_count = len(os.listdir(os.path.join(split_path, 'labels')))
        print(f"  {split}/")
        print(f"    ├── images/ ({images_count} files)")
        print(f"    └── labels/ ({labels_count} files)")

print(f"\n🏆 Dataset created using the best split found!")
print(f"   📊 Error score: {split_data['error']:.2f}")
print(f"   💾 Split details saved in: {split_file}")

📁 CREATING TRAIN/VAL/TEST FOLDER STRUCTURE FROM BEST SPLIT
✅ Using best saved split (error: 11.50)
   📅 Created: 2025-08-03T15:37:41.664310

Source: ./DATASET/dataset_by_video
Output: ./DATASET/dataset_final_split
Creating folders and copying files...

✅ Split folders created successfully using BEST SPLIT!
Files copied:
  TRAIN: 848 images
  VAL: 122 images
  TEST: 104 images

🔍 FINAL DATASET STRUCTURE:
  train/
    ├── images/ (848 files)
    └── labels/ (848 files)
  val/
    ├── images/ (122 files)
    └── labels/ (122 files)
  test/
    ├── images/ (104 files)
    └── labels/ (104 files)

🏆 Dataset created using the best split found!
   📊 Error score: 11.50
   💾 Split details saved in: ./DATASET/best_split.json

✅ Split folders created successfully using BEST SPLIT!
Files copied:
  TRAIN: 848 images
  VAL: 122 images
  TEST: 104 images

🔍 FINAL DATASET STRUCTURE:
  train/
    ├── images/ (848 files)
    └── labels/ (848 files)
  val/
    ├── images/ (122 files)
    └── labels/ (122

In [18]:
# Import and create the data.yaml file
from utils import create_data_yaml

print("📝 CREATING DATA.YAML CONFIGURATION FILE")
print("=" * 50)

# Create the data.yaml file for YOLO training
dataset_path = './DATASET/dataset_final_split'
class_names = ['Net', 'Spikeball']

yaml_file = create_data_yaml(dataset_path, class_names)

if yaml_file:
    print(f"\n🎯 YOLO Configuration ready!")
    print(f"   📄 File: {yaml_file}")
    print(f"   🚀 Ready for training with YOLOv8!")
    
    # Display the content of the created file
    print(f"\n📋 Contents of data.yaml:")
    print("-" * 30)
    with open(yaml_file, 'r') as f:
        print(f.read())
else:
    print("❌ Failed to create data.yaml file")

📝 CREATING DATA.YAML CONFIGURATION FILE
✅ Fichier data.yaml créé avec succès: ./DATASET/dataset_final_split/data.yaml
📊 Configuration:
   - Nombre de classes: 2
   - Classes: Net, Spikeball
   - Chemin du dataset: /Users/damien/Documents/PERSO/SpikeBall/DATASET/dataset_final_split

🎯 YOLO Configuration ready!
   📄 File: ./DATASET/dataset_final_split/data.yaml
   🚀 Ready for training with YOLOv8!

📋 Contents of data.yaml:
------------------------------
path: /Users/damien/Documents/PERSO/SpikeBall/DATASET/dataset_final_split
train: train
val: val
test: test
nc: 2
names:
- Net
- Spikeball

